# Week 78, Day 2: Automating files

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w078d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from pathlib import Path

p = Path("photos/2026/holiday.beach.JPG")
print(p.name)
print(p.stem, p.suffix, p.suffixes)
print(p.parent, p.parent.name)
print(p.with_suffix(".jpg"))
print(p.with_stem("day1"))
print(p.suffix.lower() in {".jpg", ".jpeg"})

In [ ]:
from pathlib import Path
import os

root = Path("demo")
for name, text in {"a.txt": "hello", "notes/b.md": "# notes", "notes/old/c.txt": "x" * 50, ".hidden": "secret"}.items():
    path = root / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text)

for p in sorted(root.rglob("*")):
    if p.is_file() and not p.name.startswith("."):
        print(f"{p.relative_to(root).as_posix():<16}{p.stat().st_size:>4} bytes")
print("text files:", [p.name for p in sorted(root.rglob("*.txt"))])

In [ ]:
from pathlib import Path
import shutil

def free_name(path):
    if not path.exists():
        return path
    n = 1
    while True:
        candidate = path.with_name(f"{path.stem} ({n}){path.suffix}")
        if not candidate.exists():
            return candidate
        n += 1

inbox, archive = Path("inbox"), Path("archive")
for folder in (inbox, archive):
    shutil.rmtree(folder, ignore_errors=True)
    folder.mkdir()
(archive / "report.pdf").write_text("last month")
(inbox / "report.pdf").write_text("this month")
(inbox / "notes.txt").write_text("hi")

for p in sorted(inbox.iterdir()):
    target = free_name(archive / p.name)
    shutil.move(p, target)
    print(f"{p} -> {target}")
print(sorted(x.name for x in archive.iterdir()), list(inbox.iterdir()))

In [ ]:
import json, os
from pathlib import Path

def write_atomic(path, text):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(text, encoding="utf-8")
    os.replace(tmp, path)

def plan_renames(names, prefix):
    plan = [(old, f"{prefix}-{n:03d}{Path(old).suffix.lower()}") for n, old in enumerate(sorted(names), 1)]
    return [(old, new) for old, new in plan if old != new]

plan = plan_renames(["IMG_2207.JPG", "IMG_2203.JPG", "IMG_2211.jpeg"], "rome")
for old, new in plan:
    print(f"{old:<15}-> {new}")
write_atomic("plan.json", json.dumps(plan))
print(json.loads(Path("plan.json").read_text()) == [list(p) for p in plan], Path("plan.json.tmp").exists())

> **Key idea.** pathlib finds files (iterdir, glob, rglob with shell wildcards; is_file, stat().st_size and st_mtime, relative_to, as_posix, mkdir(parents=True, exist_ok=True)); sort what you find, collect before changing, and skip hidden files and folders you don't mean. shutil copies (copy2, copytree with ignore_patterns), moves (move) and deletes (rmtree); rename and move can overwrite, so find a free name first. zipfile writes archives with relative arcnames and reads them with namelist, read and extractall; check names and sizes before extracting archives you didn't make. Find duplicates by size, then by SHA-256 of the contents, read in chunks. Plan batch changes as data, check them for collisions, then show (dry run) or apply them; write important files atomically with a temporary file and os.replace.

### Exercise: What's in this folder?

Write `inventory(root)`: walk every **file** below the folder `root` (a `Path` or string), skipping any file or folder whose name starts with `.`, and return a dict from lower-case extension (such as `".jpg"`, or `""` for none) to `[count, total_bytes]`. Then `largest(root, n)`: the `n` largest files (same skipping rules) as `(relative_posix_path, size)` pairs, largest first, ties in path order.

In [ ]:
from pathlib import Path

def inventory(root):
    ___

def largest(root, n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def _make():\n    import shutil\n    from pathlib import Path\n    root = Path(\"inv\")\n    shutil.rmtree(root, ignore_errors=True)\n    for name, size in {\"a.JPG\": 300, \"b.jpg\": 100, \"docs/c.pdf\": 1000, \"docs/d.txt\": 20, \"docs/Makefile\": 5,\n                       \".git/config\": 999, \"docs/.cache/e.jpg\": 5000, \".env\": 50}.items():\n        p = root / name\n        p.parent.mkdir(parents=True, exist_ok=True)\n        p.write_bytes(b\"x\" * size)\n    return root\n\ndef test_inventory():\n    \"counts and sizes by extension\"\n    root = _make()\n    assert inventory(root) == {\".jpg\": [2, 400], \".pdf\": [1, 1000], \".txt\": [1, 20], \"\": [1, 5]}\n    assert inventory(str(root)) == inventory(root)\n\ndef test_largest():\n    \"the biggest files, hidden ones skipped\"\n    root = _make()\n    assert largest(root, 3) == [(\"docs/c.pdf\", 1000), (\"a.JPG\", 300), (\"b.jpg\", 100)]\n    assert largest(root, 10)[-1] == (\"docs/Makefile\", 5)")

### Exercise: Tidy a downloads folder

Write `organise(folder, rules)`: `rules` maps a subfolder name to a list of extensions, such as `{"Images": [".jpg", ".png"], "Documents": [".pdf"]}`. Move every file directly inside `folder` (not in subfolders, and skipping names that start with `.`) whose lower-case extension is in a rule into that subfolder (creating it), and anything else into `Other`. Never overwrite: if the name is taken, use `NAME (1).EXT`, `NAME (2).EXT` and so on. Return the list of moves as `(old_name, "Subfolder/new_name")` pairs, in sorted order of the old names.

In [ ]:
import shutil
from pathlib import Path

def organise(folder, rules):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_organise():\n    \"by type, without overwriting\"\n    import shutil\n    from pathlib import Path\n    folder = Path(\"downloads\")\n    shutil.rmtree(folder, ignore_errors=True)\n    (folder / \"Images\").mkdir(parents=True)\n    (folder / \"Images\" / \"cat.jpg\").write_text(\"old cat\")\n    for name in [\"cat.jpg\", \"Dog.PNG\", \"cv.pdf\", \"song.mp3\", \".DS_Store\", \"README\"]:\n        (folder / name).write_text(name)\n    rules = {\"Images\": [\".jpg\", \".png\"], \"Documents\": [\".pdf\"]}\n    moves = organise(folder, rules)\n    assert moves == [(\"Dog.PNG\", \"Images/Dog.PNG\"), (\"README\", \"Other/README\"), (\"cat.jpg\", \"Images/cat (1).jpg\"),\n                     (\"cv.pdf\", \"Documents/cv.pdf\"), (\"song.mp3\", \"Other/song.mp3\")]\n    assert (folder / \"Images\" / \"cat.jpg\").read_text() == \"old cat\" and (folder / \"Images\" / \"cat (1).jpg\").read_text() == \"cat.jpg\"\n    assert sorted(p.name for p in folder.iterdir()) == [\".DS_Store\", \"Documents\", \"Images\", \"Other\"]\n    assert organise(folder, rules) == []")

### Exercise: Find the duplicates

Write `file_hash(path)`: the SHA-256 hex digest of a file's bytes, read in chunks of 65536 bytes. Then `duplicates(root)`: find every group of two or more files below `root` with identical contents. Group by size first, and hash only sizes shared by more than one file. Return a list of groups, each a sorted list of relative POSIX paths, the groups sorted by their first path. Empty files count too.

In [ ]:
import hashlib
from pathlib import Path

def file_hash(path):
    ___

def duplicates(root):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def _make():\n    import shutil\n    from pathlib import Path\n    root = Path(\"dups\")\n    shutil.rmtree(root, ignore_errors=True)\n    for name, data in {\"a.txt\": b\"same\", \"b/c.txt\": b\"same\", \"b/d.txt\": b\"SAME\", \"e.bin\": b\"\\x00\" * 70000,\n                       \"f/g.bin\": b\"\\x00\" * 70000, \"h.txt\": b\"same\", \"i.txt\": b\"\", \"j/k.txt\": b\"\", \"big.dat\": b\"\\x01\" * 70000}.items():\n        p = root / name\n        p.parent.mkdir(parents=True, exist_ok=True)\n        p.write_bytes(data)\n    return root\n\ndef test_hash():\n    \"a chunked SHA-256\"\n    import hashlib\n    root = _make()\n    assert file_hash(root / \"e.bin\") == hashlib.sha256(b\"\\x00\" * 70000).hexdigest()\n    assert file_hash(root / \"i.txt\") == hashlib.sha256(b\"\").hexdigest()\n\ndef test_duplicates():\n    \"groups of identical files\"\n    root = _make()\n    assert duplicates(root) == [[\"a.txt\", \"b/c.txt\", \"h.txt\"], [\"e.bin\", \"f/g.bin\"], [\"i.txt\", \"j/k.txt\"]]\n\ndef test_only_needed_hashes():\n    \"files with a unique size are never hashed\"\n    import hashlib\n    root = _make()\n    (root / \"unique.txt\").write_bytes(b\"nobody else is this long\")\n    real, seen = hashlib.sha256, []\n    class Spy:\n        def __init__(self, *a):\n            self.h = real(*a)\n            seen.append(self)\n        def update(self, b):\n            self.h.update(b)\n        def hexdigest(self):\n            return self.h.hexdigest()\n    hashlib.sha256 = Spy\n    try:\n        duplicates(root)\n    finally:\n        hashlib.sha256 = real\n    assert len(seen) == 9, f\"hashed {len(seen)} files; only the 9 with a shared size need hashing\"")

### Exercise: Zip a project

Write `backup(root, zip_path, exclude=())`: put every file below `root` into a new zip archive at `zip_path` (compressed with `ZIP_DEFLATED`), with **relative POSIX names** inside, skipping any file whose relative path or any of whose path parts matches one of the `exclude` patterns (with `fnmatch.fnmatch`). Add a `MANIFEST.txt` too, written with `writestr`, holding one line `SIZE PATH` per file added, in sorted path order. Return the sorted list of paths added (not counting the manifest). Then `restore(zip_path, dest)`: extract it into `dest`, but first check every name: if any would land outside `dest` once resolved, raise `ValueError(f"unsafe path in archive: {name}")` and extract nothing.

In [ ]:
import fnmatch
import zipfile
from pathlib import Path

def backup(root, zip_path, exclude=()):
    ___

def restore(zip_path, dest):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def _make():\n    import shutil\n    from pathlib import Path\n    for d in (\"proj\", \"restored\", \"evil_out\"):\n        shutil.rmtree(d, ignore_errors=True)\n    for name, text in {\"main.py\": \"print(1)\", \"src/app.py\": \"x = 1\", \"src/__pycache__/app.cpython.pyc\": \"junk\",\n                       \"notes.tmp\": \"tmp\", \"data/big.csv\": \"a,b\\n\" * 100}.items():\n        p = Path(\"proj\") / name\n        p.parent.mkdir(parents=True, exist_ok=True)\n        p.write_text(text)\n\ndef test_backup():\n    \"relative names, exclusions and a manifest\"\n    import zipfile\n    _make()\n    added = backup(\"proj\", \"proj.zip\", exclude=[\"__pycache__\", \"*.tmp\"])\n    assert added == [\"data/big.csv\", \"main.py\", \"src/app.py\"]\n    with zipfile.ZipFile(\"proj.zip\") as zf:\n        assert sorted(zf.namelist()) == [\"MANIFEST.txt\", \"data/big.csv\", \"main.py\", \"src/app.py\"]\n        assert zf.read(\"MANIFEST.txt\").decode() == \"400 data/big.csv\\n8 main.py\\n5 src/app.py\\n\"\n        assert zf.getinfo(\"data/big.csv\").compress_type == zipfile.ZIP_DEFLATED\n\ndef test_restore():\n    \"extracts safe archives and refuses zip slips\"\n    import zipfile\n    from pathlib import Path\n    _make()\n    backup(\"proj\", \"proj.zip\", exclude=[\"__pycache__\", \"*.tmp\"])\n    restore(\"proj.zip\", \"restored\")\n    assert Path(\"restored/src/app.py\").read_text() == \"x = 1\"\n    with zipfile.ZipFile(\"evil.zip\", \"w\") as zf:\n        zf.writestr(\"fine.txt\", \"ok\")\n        zf.writestr(\"../escaped.txt\", \"gotcha\")\n    try:\n        restore(\"evil.zip\", \"evil_out\")\n    except ValueError as e:\n        assert str(e) == \"unsafe path in archive: ../escaped.txt\"\n    else:\n        raise AssertionError(\"a ../ name must be refused\")\n    assert not Path(\"escaped.txt\").exists() and not Path(\"evil_out/fine.txt\").exists()")

### Exercise: Plan, check, apply

Write `plan_renames(names, template)`: `names` is a list of file names; sort them, then give each the new name `template.format(n=N, stem=STEM, ext=EXT)` where `N` counts from 1, `STEM` is the name without its extension and `EXT` is the lower-case extension (with its dot). Return the plan as `(old, new)` pairs, leaving out names that don't change. Then `check_plan(plan, existing)`: `existing` is the set of names currently in the folder. Return a sorted list of problems: `"NEW: several files would get this name"` when two olds map to the same new name, and `"NEW: already exists"` when a new name is in `existing` but isn't itself one of the names being renamed away. Finally `apply_plan(folder, plan)`: rename for real in two steps, first every old name to a temporary name `.tmp-N` (in plan order, N from 1) and then each temporary name to its new name, so swaps such as `a`→`b` and `b`→`a` work. Return nothing.

In [ ]:
from pathlib import Path

def plan_renames(names, template):
    ___

def check_plan(plan, existing):
    ___

def apply_plan(folder, plan):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_plan():\n    \"new names from a template\"\n    plan = plan_renames([\"IMG_9.JPG\", \"IMG_10.jpeg\", \"trip-002.jpg\"], \"trip-{n:03d}{ext}\")\n    assert plan == [(\"IMG_10.jpeg\", \"trip-001.jpeg\"), (\"IMG_9.JPG\", \"trip-002.jpg\"), (\"trip-002.jpg\", \"trip-003.jpg\")]\n    assert check_plan(plan, {\"IMG_10.jpeg\", \"IMG_9.JPG\", \"trip-002.jpg\"}) == []\n    assert plan_renames([\"trip-001.jpg\"], \"trip-{n:03d}{ext}\") == []\n    assert plan_renames([\"a.TXT\"], \"{stem}-done{ext}\") == [(\"a.TXT\", \"a-done.txt\")]\n\ndef test_check():\n    \"collisions\"\n    plan = [(\"a.txt\", \"x.txt\"), (\"b.txt\", \"x.txt\"), (\"c.txt\", \"d.txt\"), (\"e.txt\", \"f.txt\"), (\"f.txt\", \"e.txt\")]\n    existing = {\"a.txt\", \"b.txt\", \"c.txt\", \"d.txt\", \"e.txt\", \"f.txt\"}\n    assert check_plan(plan, existing) == [\"d.txt: already exists\", \"x.txt: several files would get this name\"]\n    assert check_plan([(\"a\", \"b\")], {\"a\"}) == []\n\ndef test_apply():\n    \"a swap works\"\n    import shutil\n    from pathlib import Path\n    folder = Path(\"ren\")\n    shutil.rmtree(folder, ignore_errors=True)\n    folder.mkdir()\n    for name in (\"a.txt\", \"b.txt\", \"c.txt\"):\n        (folder / name).write_text(name)\n    apply_plan(folder, [(\"a.txt\", \"b.txt\"), (\"b.txt\", \"a.txt\"), (\"c.txt\", \"z.txt\")])\n    assert {p.name: p.read_text() for p in folder.iterdir()} == {\"a.txt\": \"b.txt\", \"b.txt\": \"a.txt\", \"z.txt\": \"c.txt\"}")

### Exercise: Keep the newest backups

Write `write_atomic(path, text)`: write `text` to `PATH.tmp` (the same name plus `.tmp`, in the same folder), then move it over `path` with `os.replace`. Then `rotate(folder, pattern, keep)`: among the files in `folder` whose names match `pattern` (with `glob`), keep the `keep` newest by modification time (ties broken by name, later names counting as newer) and delete the rest. Return the sorted names deleted.

In [ ]:
import os
from pathlib import Path

def write_atomic(path, text):
    ___

def rotate(folder, pattern, keep):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_atomic():\n    \"a temporary file, then os.replace\"\n    import os\n    from pathlib import Path\n    calls = []\n    real = os.replace\n    def spy(a, b):\n        calls.append((Path(a).name, Path(b).name))\n        return real(a, b)\n    os.replace = spy\n    try:\n        write_atomic(\"settings.json\", '{\"v\": 1}')\n        write_atomic(\"settings.json\", '{\"v\": 2}')\n    finally:\n        os.replace = real\n    assert Path(\"settings.json\").read_text() == '{\"v\": 2}'\n    assert calls == [(\"settings.json.tmp\", \"settings.json\")] * 2\n\ndef test_rotate():\n    \"keep the newest\"\n    import os, shutil\n    from pathlib import Path\n    folder = Path(\"backups\")\n    shutil.rmtree(folder, ignore_errors=True)\n    folder.mkdir()\n    times = {\"db-a.zip\": 100, \"db-b.zip\": 400, \"db-c.zip\": 300, \"db-d.zip\": 300, \"db-e.zip\": 200, \"notes.txt\": 1}\n    for name, t in times.items():\n        (folder / name).write_text(name)\n        os.utime(folder / name, (t, t))\n    assert rotate(folder, \"db-*.zip\", 3) == [\"db-a.zip\", \"db-e.zip\"]\n    assert sorted(p.name for p in folder.iterdir()) == [\"db-b.zip\", \"db-c.zip\", \"db-d.zip\", \"notes.txt\"]\n    assert rotate(folder, \"db-*.zip\", 3) == []")

### Bug hunt: No photos found

The script should count the photos, but `suffix` includes the dot and keeps its case, so comparing it with `"jpg"` never matches and this prints `0 photos`. Fix the comparison so it prints `3 photos`.

In [ ]:
from pathlib import Path

names = ["a.jpg", "b.JPG", "c.png", "d.jpg", "jpg.txt"]
photos = [n for n in names if Path(n).suffix == "jpg"]
print(len(photos), "photos")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"suffix has a dot\"\n    expect_output(\"3 photos\")")

### Bug hunt: A rename that changed the folder

To change `.txt` files into `.md`, this replaces the text `.txt` anywhere in the path, which also renames part of the folder name and prints `notes.md.d/todo.md`. Change only the file's own extension, so it prints `notes.txt.d/todo.md`.

In [ ]:
from pathlib import Path

p = Path("notes.txt.d/todo.txt")
new = Path(str(p).replace(".txt", ".md"))
print(new.as_posix())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"with_suffix\"\n    expect_output(\"notes.txt.d/todo.md\")")

### Bug hunt: An archive full of absolute paths

The backup stores each file under the full path it was given, so the archive's names include the whole temporary folder's path and this prints `False`. Store each file under its path **relative to the project**, so it prints `True`.

In [ ]:
import zipfile, tempfile
from pathlib import Path

root = Path(tempfile.mkdtemp())
(root / "lib").mkdir()
(root / "app.py").write_text("x")
(root / "lib" / "util.py").write_text("y")

with zipfile.ZipFile("b.zip", "w") as zf:
    for p in sorted(root.rglob("*.py")):
        zf.write(p)
with zipfile.ZipFile("b.zip") as zf:
    names = zf.namelist()
print(names == ["app.py", "lib/util.py"])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"arcname\"\n    expect_output(\"True\")")

### Puzzle: An atomic write

Arrange a function that writes a file so a crash can never leave it half-written. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # Path("a/b.tar.gz").suffix
q = ___   # Path("a/b.tar.gz").stem
r = ___   # True or False: rglob returns files in sorted order
s = ___   # the shutil function that deletes a whole folder tree
t = ___   # the os function that swaps a finished temporary file into place

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \".gz\" and q == \"b.tar\", \"p and q\"\n    assert r is False and s == \"rmtree\" and t == \"replace\", \"r, s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a photo importer

Write `main(argv, out)` for `import-photos` (`prog="import-photos"`), with positional `source` and `dest` folders, `--dry-run`, and `--prefix` (default `"photo"`). It finds every file below `source` (recursively, skipping names starting with `.`) whose lower-case extension is `.jpg`, `.jpeg`, `.png` or `.heic`, and plans to copy each (with `shutil.copy2`) to `dest/YYYY/MM/PREFIX-YYYYMMDD-HHMMSS.EXT`, using its modification time in UTC and its lower-case extension, with `.jpeg` written as `.jpg`. Files are handled in order of modification time, then relative path. Skip a file if `dest` already holds a file with the same SHA-256 anywhere below it (or one planned earlier in this run): print `skip PATH (already imported)` with its path relative to `source`. If the target name is taken (by a different file), add `-1`, `-2` and so on before the extension. For each planned copy print `copy PATH -> TARGET` (relative to `source` and to `dest`, with `/`), prefixed with `would ` in a dry run, in which nothing is copied or created. Finish with `N copied, M skipped` (or `N to copy, M skipped` in a dry run) and return `0`.

In [ ]:
import argparse
import hashlib
import shutil
from datetime import datetime, timezone
from pathlib import Path

PHOTO = {".jpg", ".jpeg", ".png", ".heic"}

def main(argv, out):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def _make():\n    import os, shutil\n    from pathlib import Path\n    for d in (\"cam\", \"lib\"):\n        shutil.rmtree(d, ignore_errors=True)\n    t1 = 1767225600 + 3600 * 9      # 2026-01-01 09:00:00 UTC\n    t2 = 1772323200 + 45            # 2026-03-01 00:00:45 UTC\n    files = {\"DCIM/IMG_1.JPG\": (b\"one\", t1), \"DCIM/IMG_2.jpeg\": (b\"two\", t1), \"DCIM/.thumb.jpg\": (b\"x\", t1),\n             \"DCIM/clip.mp4\": (b\"vid\", t1), \"DCIM/old/IMG_0.png\": (b\"zero\", t2), \"copy_of_1.jpg\": (b\"one\", t2),\n             \"IMG_3.heic\": (b\"three\", t2)}\n    for name, (data, t) in files.items():\n        p = Path(\"cam\") / name\n        p.parent.mkdir(parents=True, exist_ok=True)\n        p.write_bytes(data)\n        os.utime(p, (t, t))\n    Path(\"lib/2026/03\").mkdir(parents=True)\n    Path(\"lib/2026/03/photo-20260301-000045.heic\").write_bytes(b\"three\")\n\ndef test_dry_run():\n    \"a plan, and nothing changed\"\n    import io\n    from pathlib import Path\n    _make()\n    out = io.StringIO()\n    assert main([\"cam\", \"lib\", \"--dry-run\"], out) == 0\n    assert out.getvalue().splitlines() == [\n        \"would copy DCIM/IMG_1.JPG -> 2026/01/photo-20260101-090000.jpg\",\n        \"would copy DCIM/IMG_2.jpeg -> 2026/01/photo-20260101-090000-1.jpg\",\n        \"would copy DCIM/old/IMG_0.png -> 2026/03/photo-20260301-000045.png\",\n        \"skip IMG_3.heic (already imported)\",\n        \"skip copy_of_1.jpg (already imported)\",\n        \"3 to copy, 2 skipped\"]\n    assert sorted(p.as_posix() for p in Path(\"lib\").rglob(\"*\") if p.is_file()) == [\"lib/2026/03/photo-20260301-000045.heic\"]\n\ndef test_import():\n    \"copies, and a second run copies nothing\"\n    import io, os\n    from pathlib import Path\n    _make()\n    out = io.StringIO()\n    assert main([\"cam\", \"lib\", \"--prefix\", \"trip\"], out) == 0\n    assert out.getvalue().splitlines()[0] == \"copy DCIM/IMG_1.JPG -> 2026/01/trip-20260101-090000.jpg\"\n    assert out.getvalue().splitlines()[-1] == \"3 copied, 2 skipped\"\n    got = Path(\"lib/2026/01/trip-20260101-090000-1.jpg\")\n    assert got.read_bytes() == b\"two\"\n    out = io.StringIO()\n    main([\"cam\", \"lib\"], out)\n    assert out.getvalue().splitlines()[-1] == \"0 copied, 5 skipped\"")

## Warm-up

A `Path` knows the parts of a file name, so you never need to split strings on `/` or `.` yourself. Notice `suffix` is only the **last** extension, includes the dot, and keeps its case: comparing it with `".jpg"` needs `.lower()`.

## Finding files

`pathlib.Path` (Week 20) is the modern way to work with files. The parts you'll use most in automation:

| You want | Write |
|---|---|
| the files directly in a folder | `folder.iterdir()` (files and folders, in no particular order) |
| matching a pattern in one folder | `folder.glob("*.csv")` |
| matching anywhere below it | `folder.rglob("*.csv")` (the same as `glob("**/*.csv")`) |
| what it is | `p.is_file()`, `p.is_dir()`, `p.exists()` |
| its size and modification time | `p.stat().st_size` (bytes), `p.stat().st_mtime` (seconds since 1970) |
| a path relative to another | `p.relative_to(root)` |
| a path with `/` on every system | `p.as_posix()` |
| make folders | `p.mkdir(parents=True, exist_ok=True)` |

Patterns use **shell-style wildcards**: `*` (any characters except `/`), `?` (one character), `[abc]` (one of these). The `fnmatch` module matches the same patterns against plain strings: `fnmatch.fnmatch("report.TMP", "*.tmp")` (case-insensitive on Windows only; use `fnmatchcase` or lower-case both for the same result everywhere).

Three habits keep folder-walking scripts correct:

- **sort** what `iterdir`, `glob` and `rglob` give you (`sorted(...)`), because their order isn't defined, and reports and tests need a stable one;
- **collect first, then change**: `files = sorted(folder.rglob("*"))` before you move or rename anything, so the walk doesn't see the folder changing under it;
- **skip what you don't mean to touch**: hidden files (names starting with `.`), the output folder, and folders, unless you mean them (`if p.is_file()`).

## Copying, moving and deleting

`pathlib` reads and writes files; **`shutil`** ("shell utilities") does the heavier jobs:

| Job | Code | Notes |
|---|---|---|
| copy a file | `shutil.copy2(src, dst)` | keeps the modification time; `dst` may be a folder |
| copy a folder tree | `shutil.copytree(src, dst, ignore=shutil.ignore_patterns("*.tmp"))` | `dst` must not exist (or pass `dirs_exist_ok=True`) |
| move or rename | `shutil.move(src, dst)` or `path.rename(new)` | `rename` **overwrites** an existing file on Linux and macOS |
| delete a file | `path.unlink()` (`missing_ok=True` to ignore a missing one) | there's no recycle bin |
| delete a folder tree | `shutil.rmtree(path)` | everything inside, at once |
| free disk space | `shutil.disk_usage(path)` | total, used and free, in bytes |

These are **dangerous** in a loop: a rename that overwrites, or an `rmtree` on the wrong variable, destroys data with no undo. So scripts that move files should never assume the target is free. A common rule is to **find a free name** by adding a number: `photo.jpg`, then `photo (1).jpg`, `photo (2).jpg`.

## Archives

The **`zipfile`** module reads and writes `.zip` files, the format every system opens:

```python
import zipfile
with zipfile.ZipFile("backup.zip", "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write("notes/a.txt", arcname="notes/a.txt")   # from a file on disk
    zf.writestr("README.txt", "made by backup.py")    # from a string
with zipfile.ZipFile("backup.zip") as zf:
    print(zf.namelist())                             # the names inside
    text = zf.read("README.txt").decode()            # one file's bytes
    zf.extractall("restored")                        # everything
```

The **`arcname`** is the name inside the archive: make it **relative** (and use `/`), or the archive contains your whole folder path. Two cautions when **extracting** archives you didn't make: a name like `../../etc/passwd` tries to escape the target folder (a "zip slip"), so check that every resolved target stays inside it; and a tiny archive can expand to gigabytes, so check sizes (`zf.infolist()` gives each `file_size`) before extracting. `shutil.make_archive("backup", "zip", root_dir="project")` zips a whole folder in one call, and the `tarfile` module does the same for `.tar.gz`.

**Duplicates.** Two files are duplicates if their **contents** are the same, whatever their names. Hash each file's bytes (`hashlib.sha256`, Week 73) and group equal hashes. Hashing everything is slow on a big disk, so first group by **size** (files of different sizes can't be equal) and hash only the groups with more than one file. Read big files in chunks (`for chunk in iter(lambda: f.read(65536), b"")`) so a 4 GB video doesn't fill memory.

## Safe batch changes

A script that renames three hundred files should be built like a careful person would do the job: **plan, check, then act**.

1. **Plan**: work out every change as data, such as a list of `(old, new)` pairs, without touching anything.
2. **Check** the plan: two files mapped to the same new name, a new name that already exists and isn't being moved away, names that don't change.
3. **Show** it for a `--dry-run`, or **apply** it.

Because the plan is plain data, it's easy to print, test and confirm, and the dangerous part is a short loop at the end.

**Atomic writes.** If a script crashes (or the power fails) halfway through writing `settings.json`, the file is left half-written and the next run can't read it. Write to a **temporary file in the same folder**, then **`os.replace(tmp, target)`**, which swaps the new file in in one step: anyone reading the file sees the old version or the new one, never half of each.

### On your own computer

Test file scripts on a **copy** of the real folder first, keep a `--dry-run` that's on by default for anything destructive, and log every change (old name, new name) so it can be undone. To watch a folder and react when files appear, the third-party `watchdog` library tells you about changes as they happen; Thursday's lesson covers running a script on a schedule instead.

## Practice

## Quiz

## Daily challenge

## Before you go

Pick one folder on your own computer that's always a mess (Downloads is the usual one) and run `inventory` and `duplicates` on it, read-only, to see what's there. Tomorrow: **scheduling jobs**: running scripts at set times, cron expressions, and what to do when a job fails at three in the morning.